# 🚀 Full LLM Fine-Tuning on 100% Dataset (AMD MI300X)

This notebook provides a complete pipeline for fine-tuning an LLM on **100% of your Java Vulnerability Dataset**.
It is optimized for your **AMD MI300X (192GB VRAM)** and supports both **Qwen2.5-Coder-32B-Instruct** and **Qwen2.5-Coder-7B-Instruct**.

**Note:** This notebook automatically filters out any truncated/corrupted code examples from the dataset before training to ensure the model outputs complete, working Java fixes.

In [ ]:
# ─── Cell 1: Install Dependencies ─────────────────────────────────────────────
!pip install -U transformers peft trl datasets accelerate wandb
!pip install bitsandbytes

In [ ]:
# ─── Cell 2: Imports ──────────────────────────────────────────────────────────
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import wandb

In [ ]:
# ─── Cell 3: Configuration ────────────────────────────────────────────────────
# Choose your settings here:
MODEL_SIZE = "32B"  # Options: "32B" or "7B"
USE_4BIT = False    # Set to True for 4-bit QLoRA, or False for native bfloat16 LoRA (recommended for ROCm)
EPOCHS = 1          # Set to 1 or 3 epochs for full training run

In [ ]:
# ─── Cell 4: Load and Clean the Dataset ───────────────────────────────────────
data_files = {
    "train": "train.jsonl",
    "validation": "val.jsonl",
    "test": "test.jsonl"
}
raw_dataset = load_dataset("json", data_files=data_files)

print(f"Original Train Dataset size: {len(raw_dataset['train'])}")

# Clean logic: Discard any examples where the target output ends mid-code (missing closing ``` backticks)
def is_complete_example(example):
    completion = (example.get('completion', '') or example.get('output', '') or '').strip()
    # If it has a code block but no closing backticks, it means the database truncated it
    if "```java" in completion:
        parts = completion.split("```java")
        if len(parts) >= 2 and "```" not in parts[1]:
            return False
    return True

dataset = raw_dataset.filter(is_complete_example)
print(f"Cleaned Train Dataset size: {len(dataset['train'])} (Removed truncated examples)")

In [ ]:
# ─── Cell 5: Apply Tokenization & Chat Template ───────────────────────────────
if MODEL_SIZE == "32B":
    model_id = "Qwen/Qwen2.5-Coder-32B-Instruct"
else:
    model_id = "Qwen/Qwen2.5-Coder-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

def format_chat_template(example):
    if "messages" in example:
        example["text"] = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    else:
        instruction = example.get('prompt', '') or example.get('instruction', '')
        input_code = example.get('input', '')
        completion = example.get('completion', '') or example.get('output', '')
        vulnerability_type = example.get('vulnerability_type', 'Vulnerable')
        
        # Combine instruction and input code into the user prompt
        full_prompt = f"{instruction}\n\n{input_code}" if input_code else instruction
        
        # Explicitly label secure/safe code
        if vulnerability_type == "Safe" or (input_code and input_code.strip() == completion.strip()):
            completion = "This Java code is completely secure and contains no vulnerabilities. No changes are required."
        else:
            # Try to restructure the explanation and code block into structured markdown sections
            parts = completion.split("```java")
            if len(parts) >= 2:
                explanation = parts[0].strip()
                code_block = parts[1].split("```")[0].strip()
                completion = (
                    f"### 🛡️ Vulnerability Analysis\n"
                    f"*   **Status**: VULNERABLE\n"
                    f"*   **Type**: {vulnerability_type}\n"
                    f"*   **Severity**: HIGH\n\n"
                    f"### 📝 Explanation\n"
                    f"{explanation}\n\n"
                    f"### 🛠️ Fixed Code\n"
                    f"```java\n"
                    f"{code_block}\n"
                    f"```"
                )
            
        messages = [
            {"role": "user", "content": full_prompt},
            {"role": "assistant", "content": completion}
        ]
        example["text"] = tokenizer.apply_chat_template(messages, tokenize=False)
    return example

formatted_dataset = dataset.map(format_chat_template)
print("Sample formatted text:")
print(formatted_dataset['train'][0]['text'])

In [ ]:
# ─── Cell 6: Load Model (ROCm Optimized) ──────────────────────────────────────
if USE_4BIT:
    print(f"Loading {MODEL_SIZE} model with 4-bit quantization...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)
else:
    print(f"Loading {MODEL_SIZE} model in native bfloat16...")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype=torch.bfloat16
    )
    model.config.use_cache = False

print("Model loaded successfully!")

In [ ]:
# ─── Cell 7: Setup LoRA Configuration ─────────────────────────────────────────
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

In [ ]:
# ─── Cell 8: Configure SFTConfig (No Step Limit + Sequence Packing) ───────────
# Set batch size and gradient accumulation to prevent VRAM OOM on the 32B model
if MODEL_SIZE == "32B" and not USE_4BIT:
    batch_size = 8
    grad_accum = 4
else:
    batch_size = 16
    grad_accum = 2

training_args = SFTConfig(
    output_dir="./large-java-vuln-model-full",
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    optim="paged_adamw_32bit",
    save_strategy="epoch",               # Save checkpoint at the end of each epoch
    eval_strategy="epoch",               # Evaluate at the end of each epoch
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=True,                           # bfloat16 for AMD hardware acceleration
    max_grad_norm=0.3,
    num_train_epochs=EPOCHS,             # Run full training epochs on 100% of data
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none",                    # Change to "wandb" if using weights and biases
    dataset_text_field="text",
    max_seq_length=2048,                 # Explicitly set max seq length to avoid truncation during packing
    packing=True                         # Enable sequence packing to decrease training time significantly
)

In [ ]:
# ─── Cell 9: Start Full Training Run ──────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset['train'],
    eval_dataset=formatted_dataset['validation'],
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

trainer.model.print_trainable_parameters()
print("Starting full training loop on 100% of the dataset...")
trainer.train()

In [ ]:
# ─── Cell 10: Save Adapter Weights ────────────────────────────────────────────
adapter_output_path = f"./java-vuln-adapter-{MODEL_SIZE.lower()}-full"
trainer.model.save_pretrained(adapter_output_path)
tokenizer.save_pretrained(adapter_output_path)
print(f"Training Complete! Full adapter saved to '{adapter_output_path}'")